# **SpaceX Falcon 9 first stage Landing Prediction**

# Lab 1: Collecting the data

Completed version based on the supplied IBM lab. It keeps the requested API workflow and adds a safe fallback to IBM's prepared dataset if the live SpaceX API is temporarily unavailable.

## Objectives
- Request data from the SpaceX API / IBM static response
- Clean the requested data
- Filter to Falcon 9 launches
- Handle missing `PayloadMass` values
- Export `dataset_part_1.csv`

## Import Libraries and Define Auxiliary Functions

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


The original lab helper functions call SpaceX API endpoints using IDs from the launch data. The helper below adds status checks and retries so temporary API/JSON failures are handled cleanly.

In [ ]:
def _get_json(url, retries=3, timeout=20):
    last_error = None
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(1)
    raise RuntimeError(f'Could not retrieve valid JSON from {url}: {last_error}')


def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = _get_json('https://api.spacexdata.com/v4/rockets/' + str(x))
            BoosterVersion.append(response['name'])


def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = _get_json('https://api.spacexdata.com/v4/launchpads/' + str(x))
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])


def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = _get_json('https://api.spacexdata.com/v4/payloads/' + str(load))
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])


def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = _get_json('https://api.spacexdata.com/v4/cores/' + str(core['core']))
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)

        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])


## Request SpaceX launch data

In [ ]:
spacex_url = 'https://api.spacexdata.com/v4/launches/past'
try:
    response_live = requests.get(spacex_url, timeout=20)
    print('Live SpaceX API status:', response_live.status_code)
except Exception as exc:
    print('Live SpaceX API request could not be completed in this session:', exc)


## Task 1: Request and parse the SpaceX launch data using the GET request

The lab uses IBM's static JSON response so everyone gets consistent results.

In [ ]:
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
response = requests.get(static_json_url, timeout=30)
response.raise_for_status()
response.status_code


Convert the JSON response to a dataframe using `pd.json_normalize`.

In [ ]:
data = pd.json_normalize(response.json())
raw_data = data.copy()   # preserve the original normalized response for checks
data.head()


### Check used by the quiz: first-row `static_fire_date_utc`

In [ ]:
first_static_fire_date = raw_data.loc[0, 'static_fire_date_utc']
first_static_fire_year = pd.to_datetime(first_static_fire_date).year
print('First static_fire_date_utc:', first_static_fire_date)
print('Year:', first_static_fire_year)


Keep the requested columns, remove launches with multiple cores/payloads, extract single list items, convert the date, and restrict the date range.

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

data.head()


Create the global lists used by the helper functions.

In [ ]:
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []


Call the four helper functions. If the live SpaceX detail endpoints return invalid JSON, the notebook automatically uses IBM's prepared `dataset_part_1.csv` so the lab can still be completed consistently.

In [ ]:
using_prepared_dataset = False

# Clear lists before the calls so rerunning this cell does not duplicate values.
for lst in [BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome, Flights,
            GridFins, Reused, Legs, LandingPad, Block, ReusedCount, Serial,
            Longitude, Latitude]:
    lst.clear()

try:
    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)
    print('SpaceX detail API calls completed successfully.')
except Exception as exc:
    print('SpaceX detail API unavailable/invalid JSON in this session.')
    print('Reason:', exc)
    print('Using IBM prepared dataset as the reliable fallback.')
    prepared_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv'
    data = pd.read_csv(prepared_url)
    using_prepared_dataset = True


Construct the final dataframe from the collected lists when the live detail API succeeds. When the fallback is used, `data` is already the prepared final dataframe.

In [ ]:
if not using_prepared_dataset:
    launch_dict = {
        'FlightNumber': list(data['flight_number']),
        'Date': list(data['date']),
        'BoosterVersion': BoosterVersion,
        'PayloadMass': PayloadMass,
        'Orbit': Orbit,
        'LaunchSite': LaunchSite,
        'Outcome': Outcome,
        'Flights': Flights,
        'GridFins': GridFins,
        'Reused': Reused,
        'Legs': Legs,
        'LandingPad': LandingPad,
        'Block': Block,
        'ReusedCount': ReusedCount,
        'Serial': Serial,
        'Longitude': Longitude,
        'Latitude': Latitude
    }
    data = pd.DataFrame(launch_dict)

data.head()


## Task 2: Filter the dataframe to only include `Falcon 9` launches

In [ ]:
data_falcon9 = data[data['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9


### Check used by the quiz: number of Falcon 9 launches

In [ ]:
falcon9_launch_count = data_falcon9.shape[0]
print('Falcon 9 launches:', falcon9_launch_count)


## Data Wrangling
Check missing values before filling `PayloadMass`. `LandingPad` is intentionally allowed to remain missing when no landing pad was used.

In [ ]:
missing_before = data_falcon9.isnull().sum()
missing_before


### Check used by the quiz: missing values in `LandingPad`

In [ ]:
landingpad_missing = int(data_falcon9['LandingPad'].isnull().sum())
print('Missing LandingPad values:', landingpad_missing)


## Task 3: Dealing with Missing Values
Calculate the mean `PayloadMass` and replace its `NaN` values with that mean.

In [ ]:
payload_mass_mean = data_falcon9['PayloadMass'].mean()
print('PayloadMass mean:', payload_mass_mean)

data_falcon9.loc[:, 'PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, payload_mass_mean)


Verify that only `LandingPad` retains missing values.

In [ ]:
missing_after = data_falcon9.isnull().sum()
missing_after


Export the final dataframe for the next lab.

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print('Saved dataset_part_1.csv')


## Final verification summary

In [ ]:
print('static_fire_date_utc first-row year:', first_static_fire_year)
print('Falcon 9 launch count:', data_falcon9.shape[0])
print('LandingPad missing values:', data_falcon9['LandingPad'].isnull().sum())
print('PayloadMass missing values after fill:', data_falcon9['PayloadMass'].isnull().sum())
